# 20 — Readout Leakage Benchmarking

## Purpose

Quantify measurement-induced state transitions (leakage) caused by the readout pulse. Repeated measurements can drive the qubit out of the computational subspace or cause photon-number changes in the storage cavity. This notebook benchmarks the QND-ness of the calibrated readout.

## Methodology

1. Prepare the qubit in a known state
2. Apply a variable number of consecutive readout pulses (1, 5, 10, 20, 50)
3. After the readout burst, measure the final qubit state to detect populations outside the preparation state
4. Plot leakage probability vs. number of readout repetitions

## Expected Outcomes

- Leakage probability increasing approximately linearly with number of readouts
- Per-readout leakage rate < 0.1% for a well-optimized readout
- Results inform whether active reset (notebook 18) is needed between measurement rounds

## Prerequisites

- **Notebook 16** — readout calibration complete (readout parameters must be finalized)
- **Notebook 05** — qubit π-pulse calibrated (for state preparation and verification)

## 1. Setup — Session Bootstrap

Open the notebook stage and load the readout calibration checkpoint from notebook 16.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="20_readout_leakage_benchmarking",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

readout_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="16_readout_calibration",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if readout_checkpoint is not None:
    print(f"Prior stage 16 status: {readout_checkpoint['status']}")

## 2. Configuration — Leakage Benchmarking Defaults

Set the averaging count and the list of consecutive readout repetitions to sweep.

In [ ]:
LEAKAGE_N_AVG = 50000
LEAKAGE_N_READOUTS = [1, 5, 10, 20, 50]

print("Readout leakage benchmarking settings:")
print(f"  n_avg: {LEAKAGE_N_AVG}")
print(f"  n_readouts sweep: {LEAKAGE_N_READOUTS}")

## 3. Execution — Readout-Induced Leakage Measurement

For each readout count in the sweep, apply N consecutive readout pulses and measure the resulting qubit state to detect measurement-induced transitions.

In [ ]:
leakage_results = {}

for n_readouts in LEAKAGE_N_READOUTS:
    result = session.readout_leakage_measurement(
        n_readouts=n_readouts,
        n_avg=LEAKAGE_N_AVG,
    )
    leakage_results[n_readouts] = result
    print(f"  n_readouts={n_readouts}: done")

print("Readout leakage sweep complete.")

## 4. Analysis — Leakage Summary Plot

Plot leakage probability vs. number of consecutive readouts. A linear slope gives the per-readout leakage rate.

In [ ]:
# Plot leakage probability vs. number of readouts
n_ro_list = sorted(leakage_results.keys())

fig, ax = plt.subplots(figsize=(8, 5))
# Placeholder: fill with actual leakage metrics from results
ax.set_xlabel("Number of consecutive readouts")
ax.set_ylabel("Leakage probability")
ax.set_title("Readout-Induced Leakage")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Leakage summary plotted (update with actual metrics from results).")

## 5. Checkpoint — Save Leakage Benchmarking Results

Record characterization results. No calibrations are applied — this is a diagnostic notebook that informs whether readout parameters need revision.

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="20_readout_leakage_benchmarking",
    status="characterized",
    summary="Readout-induced leakage benchmarked across multiple consecutive readout counts.",
    consumed_inputs={
        "leakage_n_avg": LEAKAGE_N_AVG,
        "leakage_n_readouts": LEAKAGE_N_READOUTS,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="21_storage_cavity_characterization",
    notes=[
        "Characterization-only — no calibrations applied.",
        "Results inform whether active reset is needed after measurement.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")